# 4a. Revenue quality

**What an investor reads into this.** A buyer of a SaaS business pays a multiple of recurring revenue, because recurring revenue is what repeats next year without a new sale. One-off revenue is valued far lower. So the first question is how much of Northbridge's revenue and growth is recurring, and whether headline growth is being flattered by one-off billing. ARR at each year end shows the run-rate the buyer inherits.

Definitions: `docs/metric_definitions.md`. Before any analysis, section 0 checks the two possible sources of MRR against each other.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from nb_utils import connect, q, sql_file, save_table, md_table, gbp, pct, DOCS, replace_block
import charts as ch

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
con = connect()

## 0. Subscription MRR against invoiced MRR

In [2]:
svi = save_table(sql_file(con, "sub_vs_invoice_mrr"), "4a_sub_vs_invoice_mrr")
pcd = save_table(sql_file(con, "price_change_distribution"), "4a_price_change_distribution")
k = save_table(sql_file(con, "uplift_factors"), "4a_uplift_factors")
assert (svi.unmatched_lines == 0).all(), "subscription and invoice lines disagree"
assert (svi.lines_at_that_ratio == svi.invoiced_lines).all(), "the uplift is not uniform across lines"
display(svi); display(pcd); display(k)

,year_end,subscription_lines,invoiced_lines,unmatched_lines,subscription_mrr,invoiced_mrr,gap,invoiced_over_subscription,most_common_line_ratio,lines_at_that_ratio
0,2022-12-01,1566,1566,0,2417140.30,2417140.30,0.00,1.0000,1.0000,1566
1,2023-12-01,1893,1893,0,2731778.55,2731778.55,0.00,1.0000,1.0000,1893
2,2024-12-01,2149,2149,0,2975939.12,3124736.34,148797.22,1.0500,1.0500,2149
3,2025-12-01,2355,2355,0,3067780.81,3446652.32,378871.51,1.1235,1.1235,2355


,year,pct_change,lines,share_of_lines_pct
0,2023,0.0,1425,100.0
1,2024,5.0,1686,100.0
2,2025,7.0,1890,100.0


,year,k,share_of_lines_at_k_pct,continuing_lines
0,2023,1.00,100.0,1425
1,2024,1.05,100.0,1686
2,2025,1.07,100.0,1890


In [3]:
replace_block(DOCS / "metric_definitions.md", "sub_vs_invoice", md_table(
    svi[["year_end", "subscription_lines", "invoiced_lines", "unmatched_lines", "subscription_mrr", "invoiced_mrr",
         "gap", "invoiced_over_subscription", "lines_at_that_ratio"]],
    {"subscription_mrr": gbp, "invoiced_mrr": gbp, "gap": gbp, "invoiced_over_subscription": lambda v: f"{v:.4f}"}))
replace_block(DOCS / "metric_definitions.md", "price_change_evidence", md_table(
    pcd, {"pct_change": lambda v: pct(v, 2), "share_of_lines_pct": pct}) + "\n\n" + md_table(
    k, {"k": lambda v: f"{v:.4f}", "share_of_lines_at_k_pct": pct}))
print("metric_definitions.md blocks updated")

metric_definitions.md blocks updated


## 1. Recurring and one-off revenue by year

In [4]:
mix = q(con, """
    SELECT year(month_start) AS year,
           sum(recurring_net) AS recurring,
           sum(oneoff_signup_net) AS signup_implementation,
           sum(oneoff_other_net) AS other_implementation,
           sum(oneoff_net) AS oneoff,
           sum(net_revenue) AS net_revenue
    FROM fact_revenue_monthly GROUP BY 1 ORDER BY 1
""")
mix["recurring_pct"] = 100 * mix.recurring / mix.net_revenue
mix["oneoff_pct"] = 100 * mix.oneoff / mix.net_revenue
mix["other_implementation_pct"] = 100 * mix.other_implementation / mix.net_revenue
save_table(mix, "4a_revenue_mix")

,year,recurring,signup_implementation,other_implementation,oneoff,net_revenue,recurring_pct,oneoff_pct,other_implementation_pct
0,2022,27248292.05,983859.59,0.00,983859.59,28232151.64,96.515109,3.484891,0.000000
1,2023,31014666.15,1178203.80,0.00,1178203.80,32192869.95,96.340172,3.659828,0.000000
2,2024,36114683.69,1144214.82,0.00,1144214.82,37258898.51,96.929016,3.070984,0.000000
3,2025,40475097.15,1204747.89,4076574.31,5281322.20,45756419.35,88.457746,11.542254,8.909295


## 2. ARR at each year end

In [5]:
arr = q(con, """
    SELECT year(month_start) AS year, 12 * sum(mrr) AS arr, count(DISTINCT customer_id) AS active_customers,
           12 * sum(mrr) / count(DISTINCT customer_id) AS arr_per_customer
    FROM fact_mrr_monthly WHERE month(month_start) = 12 GROUP BY 1 ORDER BY 1
""")
arr["arr_growth_pct"] = 100 * arr.arr.pct_change()
arr = arr.merge(k[["year", "k"]], on="year", how="left")
arr["arr_growth_ex_price_pct"] = 100 * ((arr.arr / arr.k.fillna(1)) / arr.arr.shift(1) - 1)
save_table(arr, "4a_arr")

,year,arr,active_customers,arr_per_customer,arr_growth_pct,k,arr_growth_ex_price_pct
0,2022,29005683.60,1010,28718.498614,NaN,NaN,NaN
1,2023,32781342.60,1205,27204.433693,13.016963,1.00,13.016963
2,2024,37496836.08,1343,27920.205570,14.384687,1.05,8.937797
3,2025,41359827.84,1462,28289.895923,10.302181,1.07,3.086150


## 3. Growth: headline against recurring-only and against growth excluding other implementation

In [6]:
g = mix[["year", "net_revenue", "recurring", "other_implementation"]].copy()
g["net_ex_other_implementation"] = g.net_revenue - g.other_implementation
g["headline_growth_pct"] = 100 * g.net_revenue.pct_change()
g["recurring_growth_pct"] = 100 * g.recurring.pct_change()
g["growth_ex_other_implementation_pct"] = 100 * g.net_ex_other_implementation.pct_change()
g["gap_headline_less_recurring_pts"] = g.headline_growth_pct - g.recurring_growth_pct
g["gap_headline_less_ex_other_pts"] = g.headline_growth_pct - g.growth_ex_other_implementation_pct
save_table(g, "4a_growth")

,year,net_revenue,recurring,other_implementation,net_ex_other_implementation,headline_growth_pct,recurring_growth_pct,growth_ex_other_implementation_pct,gap_headline_less_recurring_pts,gap_headline_less_ex_other_pts
0,2022,28232151.64,27248292.05,0.00,28232151.64,NaN,NaN,NaN,NaN,NaN
1,2023,32192869.95,31014666.15,0.00,32192869.95,14.029105,13.822423,14.029105,0.206683,0.00000
2,2024,37258898.51,36114683.69,0.00,37258898.51,15.736492,16.443890,15.736492,-0.707397,0.00000
3,2025,45756419.35,40475097.15,4076574.31,41679845.04,22.806688,12.073797,11.865478,10.732892,10.94121


## 4. Chart

In [7]:
fig, ax = plt.subplots(figsize=(8, 4.6))
x = mix.year.astype(str)
b1 = ax.bar(x, mix.recurring, width=0.6, color=ch.BLUE, label="Recurring", edgecolor=ch.SURFACE, linewidth=1.5)
b2 = ax.bar(x, mix.signup_implementation, bottom=mix.recurring, width=0.6, color=ch.ORANGE,
            label="Implementation at signup", edgecolor=ch.SURFACE, linewidth=1.5)
b3 = ax.bar(x, mix.other_implementation, bottom=mix.recurring + mix.signup_implementation, width=0.6, color=ch.AQUA,
            label="Other implementation", edgecolor=ch.SURFACE, linewidth=1.5)
for xi, r in zip(x, mix.itertuples()):
    ax.text(xi, r.recurring / 2, f"{r.recurring_pct:.1f}%\nrecurring", ha="center", va="center", color="white", fontsize=8, fontweight="bold")
    ax.text(xi, r.net_revenue * 1.01, gbp(r.net_revenue / 1e6, 1) + "m", ha="center", va="bottom", color=ch.TEXT, fontsize=8)
r25 = mix[mix.year == mix.year.max()].iloc[0]
ax.text(len(mix) - 1 + 0.36, r25.recurring + r25.signup_implementation + r25.other_implementation / 2,
        f"Other\nimplementation\n{gbp(r25.other_implementation / 1e6, 2)}m\n({r25.other_implementation_pct:.1f}% of 2025)",
        ha="left", va="center", fontsize=8, color=ch.TEXT_2)
ax.set_xlim(-0.5, len(mix) - 0.5 + 0.9)
ax.yaxis.set_major_formatter(ch.gbp_m(0))
ax.set_ylabel("Net revenue (£m)")
ax.set_xlabel("Calendar year")
ax.set_title("Net revenue by type, 2022 to 2025")
ax.legend(loc="upper left", ncol=3)
ax.set_ylim(0, mix.net_revenue.max() * 1.15)
print(ch.save(fig, "4a_revenue_mix"))

outputs/charts/4a_revenue_mix.png


In [8]:
display(mix); display(arr); g

,year,recurring,signup_implementation,other_implementation,oneoff,net_revenue,recurring_pct,oneoff_pct,other_implementation_pct
0,2022,27248292.05,983859.59,0.00,983859.59,28232151.64,96.515109,3.484891,0.000000
1,2023,31014666.15,1178203.80,0.00,1178203.80,32192869.95,96.340172,3.659828,0.000000
2,2024,36114683.69,1144214.82,0.00,1144214.82,37258898.51,96.929016,3.070984,0.000000
3,2025,40475097.15,1204747.89,4076574.31,5281322.20,45756419.35,88.457746,11.542254,8.909295


,year,arr,active_customers,arr_per_customer,arr_growth_pct,k,arr_growth_ex_price_pct
0,2022,29005683.60,1010,28718.498614,NaN,NaN,NaN
1,2023,32781342.60,1205,27204.433693,13.016963,1.00,13.016963
2,2024,37496836.08,1343,27920.205570,14.384687,1.05,8.937797
3,2025,41359827.84,1462,28289.895923,10.302181,1.07,3.086150


,year,net_revenue,recurring,other_implementation,net_ex_other_implementation,headline_growth_pct,recurring_growth_pct,growth_ex_other_implementation_pct,gap_headline_less_recurring_pts,gap_headline_less_ex_other_pts
0,2022,28232151.64,27248292.05,0.00,28232151.64,NaN,NaN,NaN,NaN,NaN
1,2023,32192869.95,31014666.15,0.00,32192869.95,14.029105,13.822423,14.029105,0.206683,0.00000
2,2024,37258898.51,36114683.69,0.00,37258898.51,15.736492,16.443890,15.736492,-0.707397,0.00000
3,2025,45756419.35,40475097.15,4076574.31,41679845.04,22.806688,12.073797,11.865478,10.732892,10.94121
